# Study 2: Why do agents favor high mitigation?

- Notebook 04 showed frequent high mitigation and little development after the early steps. I want to understand what the agents experience when they make those choices.
- Does development raise income enough to pay for repeated mitigation? Do agents stop developing after they have enough currency, or while currency is still growing?
- How do development gains and pollution penalties contribute to reward at different stages? Does this help explain the preference for high mitigation?

## What does the reward measure?

- A successful development action gives an immediate reward based on the increase in EDI. A higher EDI also raises future income, but income itself is not a reward term.
- Pollution reduces reward every step. Mitigation can improve reward by lowering pollution, even when the region is far from failure.
- Spending currency has no direct reward penalty; it limits which actions the region can afford later. Failed action attempts have a penalty.

## Settings

In [13]:
import json

import pandas as pd

from study_2_economic_sustainability.agents import TabularQLearner
from study_2_economic_sustainability.environment import EconomicSustainabilityEnvironment

In [14]:
REWARD_MODES = ("local", "global", "mixed")
TRAINING_SEEDS = (11, 23, 37, 41, 53)
EVALUATION_SEEDS = tuple(range(100_000, 100_020))
Q_TABLE_DIRECTORY = "../results/reward_comparison"

## Load the saved policies

- Each file contains four Q-tables, one for each region.
- I will use the same greedy action choices as notebook 04 and record what happens around each choice.

In [15]:
agents_by_run = {}

for reward_mode in REWARD_MODES:
    for training_seed in TRAINING_SEEDS:
        filename = f"{Q_TABLE_DIRECTORY}/q_{reward_mode}_{training_seed}.json"
        with open(filename, encoding="utf-8") as file:
            tables = json.load(file)

        agents = []
        for region, table in enumerate(tables):
            agent = TabularQLearner(region)
            for key, values in table.items():
                observation = tuple(int(value) for value in key.split(","))
                agent.q[observation] = values
            agents.append(agent)

        agents_by_run[(reward_mode, training_seed)] = agents

print(f"Loaded {len(agents_by_run)} trained policies")

Loaded 15 trained policies


## Record what happens at each step

- One row will describe one region at one step. The four regions still act together.
- EDI and currency are recorded before the action. Income, cost, and reward come from that step.

In [16]:
def step_rows(step, actions, edi_before, currency_before, rewards, info):
    rows = []

    for region in range(4):
        rows.append({
            "step": step,
            "region": region,
            "action": actions[region],
            "edi_before": edi_before[region],
            "currency_before": currency_before[region],
            "income": info["income"][region],
            "cost_paid": info["costs_paid"][region],
            "edi_gain": info["economic_rewards"][region],
            "pollution_reward": info["pollution_rewards"][region],
            "received_reward": rewards[region],
        })

    return rows

In [17]:
def collect_episode(agents, reward_mode, seed):
    environment = EconomicSustainabilityEnvironment(reward_mode)
    observations = environment.reset(seed)
    rows = []

    while True:
        edi_before = environment.development_indices
        currency_before = environment.currency_balances
        actions = []
        for region in range(4):
            action = agents[region].select_action(observations[region], greedy=True)
            actions.append(action)

        observations, rewards, terminated, truncated, info = environment.step(tuple(actions))
        rows.extend(step_rows(
            environment.step_count, actions, edi_before, currency_before, rewards, info
        ))
        if terminated or truncated:
            break

    return rows

In [18]:
step_data = []

for reward_mode in REWARD_MODES:
    for training_seed in TRAINING_SEEDS:
        agents = agents_by_run[(reward_mode, training_seed)]

        for environment_seed in EVALUATION_SEEDS:
            rows = collect_episode(agents, reward_mode, environment_seed)
            for row in rows:
                row["reward_mode"] = reward_mode
                row["training_seed"] = training_seed
            step_data.extend(rows)

steps = pd.DataFrame(step_data)
print(f"Recorded {len(steps):,} region-steps")

Recorded 240,000 region-steps


## What changes over the episode?

- I summarize region-steps across the 20 held-out starts within each training seed, then average the five seed summaries.

In [19]:
steps["stage"] = 1 + (steps["step"] - 1) // 50
steps["develop"] = steps["action"].eq(1)
steps["high_mitigation"] = steps["action"].eq(4)
steps["can_afford_high"] = steps["currency_before"].ge(50)
steps["currency_change"] = steps["income"] - steps["cost_paid"]

stage_by_seed = steps.groupby(["reward_mode", "training_seed", "stage"]).agg(
    develop_percent=("develop", "mean"),
    high_percent=("high_mitigation", "mean"),
    mean_edi=("edi_before", "mean"),
    mean_income=("income", "mean"),
    mean_currency=("currency_before", "mean"),
    currency_change_per_step=("currency_change", "mean"),
    can_afford_high_percent=("can_afford_high", "mean"),
    mean_edi_gain=("edi_gain", "mean"),
    mean_pollution_reward=("pollution_reward", "mean"),
    mean_received_reward=("received_reward", "mean"),
)

for column in ("develop_percent", "high_percent", "can_afford_high_percent"):
    stage_by_seed[column] *= 100

stage_summary = stage_by_seed.groupby(["reward_mode", "stage"]).mean()

### Actions and resources

- `mean_currency` is the balance before a decision.
- `currency_change_per_step` is income minus spending; a positive value means balances grow on average.
- `can_afford_high_percent` shows how often a region had at least 50 currency before acting. It does not say whether high mitigation was its best choice.

In [20]:
resource_columns = [
    "develop_percent", "high_percent", "mean_edi", "mean_income",
    "mean_currency", "currency_change_per_step", "can_afford_high_percent",
]
display(stage_summary[resource_columns].round(1))

develop_percent  high_percent  mean_edi  mean_income  \
reward_mode stage                                                         
global      1                 16.8          33.3      42.8         41.4   
            2                  1.1          66.4      58.4         49.2   
            3                  0.0          64.6      58.7         49.3   
            4                  0.0          64.5      58.7         49.3   
local       1                 21.5          54.4      55.9         47.9   
            2                  0.6          97.6      71.0         55.5   
            3                  0.0          95.1      71.0         55.5   
            4                  0.0          95.0      71.0         55.5   
mixed       1                 19.4          52.5      53.9         46.9   
            2                  0.3          96.2      68.6         54.3   
            3                  0.0          94.5      68.6         54.3   
            4                  0.0          94.1      68.6         54.3   

                   mean_currency  currency_change_per_step  \
reward_mode stage                                            
global      1              415.0                      -0.2   
            2              787.8                       9.5   
            3             1299.3                      10.7   
            4             1834.9                      10.7   
local       1              275.8                      -4.9   
            2              467.6                       5.9   
            3              787.0                       6.7   
            4             1125.6                       6.8   
mixed       1              301.7                      -4.1   
            2              496.4                       5.2   
            3              773.8                       5.9   
            4             1073.4                       6.0   

                   can_afford_high_percent  
reward_mode stage                           
global      1                        100.0  
            2                        100.0  
            3                        100.0  
            4                        100.0  
local       1                         99.9  
            2                        100.0  
            3                        100.0  
            4                        100.0  
mixed       1                         99.9  
            2                        100.0  
            3                        100.0  
            4                        100.0

- Development is mostly an early choice. In steps 1-50, agents develop in 16.8% of global, 21.5% of local, and 19.4% of mixed region-steps. By steps 51-100, those shares fall to 1.1%, 0.6%, and 0.3%. EDI and income then stay almost flat.
- High mitigation moves the other way: about 33% of global actions and 53-54% of local and mixed actions in the first stage, rising to about 65% and 94-98% later.
- Could currency explain this switch? High mitigation costs 50, but regions can afford it on essentially every recorded step, even in the first stage. Balances keep growing after development stops: by steps 151-200, mean currency reaches about 1,835 for global, 1,126 for local, and 1,073 for mixed.
- Income also covers much of the later spending. Local and mixed agents earn about 54-56 currency per step and gain about 6 after costs; global agents earn about 49 and gain about 11 because they choose a broader action mix. I wonder whether development has become less attractive once income can support the preferred mitigation pattern. These averages do not show that the agents planned for that balance.

### Reward components

- `mean_edi_gain` is the immediate positive development term. `mean_pollution_reward` is the negative pollution term. Both are local components before reward pooling.
- `mean_received_reward` is what each agent actually received under its reward mode. It can also include invalid-action and failure penalties.
- These averages describe what happened under the chosen actions. They do not tell us the reward the agent would have received for a different action in the same state.

In [21]:
reward_columns = ["mean_edi_gain", "mean_pollution_reward", "mean_received_reward"]
display(stage_summary[reward_columns].round(2))

mean_edi_gain  mean_pollution_reward  mean_received_reward
reward_mode stage                                                            
global      1               1.13                  -0.74                  0.36
            2               0.04                  -0.15                 -0.12
            3               0.00                  -0.09                 -0.09
            4               0.00                  -0.09                 -0.09
local       1               1.41                  -0.64                  0.69
            2               0.01                  -0.02                 -0.03
            3               0.00                   0.00                  0.00
            4               0.00                   0.00                  0.00
mixed       1               1.36                  -0.60                  0.74
            2               0.01                  -0.01                  0.01
            3               0.00                  -0.00                 -0.00
            4               0.00                  -0.00                 -0.00

- The positive development term is largest in steps 1-50: 1.13 per region-step for global, 1.41 for local, and 1.36 for mixed. It then approaches zero as development actions nearly disappear.
- The pollution term is negative in the first stage, then becomes small under local and mixed policies. Global retains a larger pollution penalty, about -0.09 per region-step in the last two stages. That fits the higher mitigation share and lower pollution seen for local and mixed policies.
- High mitigation has no direct reward bonus. It can help by reducing current and future pollution penalties, while its currency cost is not subtracted from reward. That could make repeated high mitigation attractive once agents can afford it. But these rows show rewards for the actions actually taken, so they cannot tell us how much better high mitigation was than medium or development in the same state.
- I would check those alternatives before deciding that the reward is too mitigation-focused. For now, the strongest observation is that development fades while currency grows and pollution penalties stay near zero under the policies that mitigate most.